In [ ]:
!pip install lime shap torchcam opencv-python matplotlib

In [ ]:

MODEL_PATH = "/content/drive/MyDrive/mobilenet_pneumonia.pth"
IMAGE_FOLDER = "/content/drive/MyDrive/CHEST X-RAY IMAGES/chest_xray/test"

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models

import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

from PIL import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
model = models.mobilenet_v2(pretrained=False)

model.classifier[1] = nn.Linear(model.last_channel, 2)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

model = model.to(device)
model.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
classes = ["Normal", "Pneumonia"]

In [ ]:
def predict_image(image_path):

    image = Image.open(image_path).convert("RGB")

    img = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img)

        probs = torch.softmax(output, dim=1)

        conf, pred = torch.max(probs,1)

    label = classes[pred.item()]
    confidence = conf.item()*100

    return label, confidence

In [ ]:
for class_folder in os.listdir(IMAGE_FOLDER):

    class_path = os.path.join(IMAGE_FOLDER, class_folder)

    # check if folder
    if os.path.isdir(class_path):

        for img_name in os.listdir(class_path):

            img_path = os.path.join(class_path, img_name)

            label, conf = predict_image(img_path)

            print(f"Image: {img_name}")
            print(f"Actual Class: {class_folder}")
            print(f"Predicted: {label}")
            print(f"Confidence: {conf:.2f}%")
            print("--------------------")

In [ ]:
from torchcam.methods import GradCAM
from torchvision.transforms.functional import to_pil_image

cam_extractor = GradCAM(model, target_layer="features.18")

In [ ]:
from torchcam.methods import GradCAM
import cv2

cam_extractor = GradCAM(model, target_layer="features.18")

def show_gradcam(image_path):

    image = Image.open(image_path).convert("RGB")

    img = transform(image).unsqueeze(0).to(device)

    img.requires_grad_()   # IMPORTANT FIX

    output = model(img)

    pred = output.argmax(dim=1).item()

    activation_map = cam_extractor(pred, output)[0].cpu()

    img_np = np.array(image.resize((224,224)))

    heatmap = cv2.resize(activation_map.numpy(), (224,224))
    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed = heatmap * 0.4 + img_np

    plt.figure(figsize=(6,6))
    plt.imshow(superimposed.astype("uint8"))
    plt.title("GradCAM Explanation")
    plt.axis("off")
    plt.show()

In [ ]:
show_gradcam("/content/drive/MyDrive/CHEST X-RAY IMAGES/chest_xray/test/NORMAL/IM-0001-0001.jpeg")

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

In [ ]:
def batch_predict(images):

    model.eval()

    batch = torch.stack([
        transform(Image.fromarray(img))
        for img in images
    ]).to(device)

    batch.requires_grad = True   # important for LIME

    outputs = model(batch)

    probs = torch.softmax(outputs, dim=1)

    return probs.detach().cpu().numpy()

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

def show_lime(image_path):

    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    explainer = lime_image.LimeImageExplainer()

    explanation = explainer.explain_instance(
        image_np,
        batch_predict,
        top_labels=2,
        hide_color=0,
        num_samples=1000
    )

    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0],
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    plt.figure(figsize=(6,6))
    plt.imshow(mark_boundaries(temp, mask))
    plt.title("LIME Explanation")
    plt.axis("off")
    plt.show()

In [ ]:
show_lime("/content/drive/MyDrive/CHEST X-RAY IMAGES/chest_xray/test/NORMAL/IM-0001-0001.jpeg")

In [ ]:
import shap

In [ ]:
background_images = []

normal_folder = "/content/drive/MyDrive/CHEST X-RAY IMAGES/chest_xray/test/NORMAL"

files = os.listdir(normal_folder)[:10]

for f in files:

    img_path = os.path.join(normal_folder, f)

    img = Image.open(img_path).convert("RGB")

    img = transform(img)

    background_images.append(img)

background = torch.stack(background_images).to(device)

In [ ]:
import shap

explainer = shap.GradientExplainer(model, background)

In [ ]:
def show_shap(image_path):

    image = Image.open(image_path).convert("RGB")

    img = transform(image).unsqueeze(0).to(device)

    shap_values = explainer.shap_values(img)

    # Convert original image to numpy
    img_np = np.array(image.resize((224,224)))

    # Add batch dimension (important)
    img_np = np.expand_dims(img_np, axis=0)

    # Convert SHAP output (C,H,W) -> (H,W,C)
    shap_val = np.transpose(shap_values[0][0], (1,2,0))

    # Add batch dimension
    shap_val = np.expand_dims(shap_val, axis=0)

    shap.image_plot(shap_val, img_np)

In [ ]:
show_shap("/content/drive/MyDrive/CHEST X-RAY IMAGES/chest_xray/test/NORMAL/IM-0001-0001.jpeg")

In [ ]:
def analyze_image(image_path):

    label, conf = predict_image(image_path)

    print("Prediction:", label)
    print(f"Confidence: {conf:.2f}%")

    print("Generating GradCAM...")
    show_gradcam(image_path)

    print("Generating LIME...")
    show_lime(image_path)

    print("Generating SHAP...")
    show_shap(image_path)

In [ ]:
analyze_image("/content/drive/MyDrive/CHEST X-RAY IMAGES/chest_xray/test/NORMAL/IM-0001-0001.jpeg")